# optimizer-repr-string — worked example 3: Multi-line __repr__ for an optimizer with param groups

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-repr-string`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When an optimizer has multiple param groups (each with possibly different hyperparameters), a flat single-line repr is insufficient. PyTorch's convention is a multi-line format: a header with the class name, one labeled block per group, and a closing parenthesis. The hparam keys within each group are sorted alphabetically.

## Worked solution

**Step 1 — Write the header line.**
The first line is `'ClassName ('`. Note the space before the opening paren — this matches PyTorch's format.

**Step 2 — Write one block per param group.**
For group `i`, we output a `'Parameter Group <i>'` line followed by sorted `'    key: value'` lines for every key except `'params'` (which holds the actual tensors).

**Step 3 — Write the closing line.**
The final line is a lone `')'`. Lines are joined with `'\n'` with no trailing newline.

**Step 4 — Verify against a manually constructed expected string.**
We build the optimizer with two groups and compare `repr(opt)` to the expected multi-line string character-by-character.

In [ ]:
import torch as t
import torch.nn as nn

class MultiGroupSGD:
    def __init__(self, param_groups):
        self.param_groups = [
            {'params': list(g['params']), **{k: v for k, v in g.items() if k != 'params'}}
            for g in param_groups
        ]

    def __repr__(self):
        lines = ['MultiGroupSGD (']
        for i, group in enumerate(self.param_groups):
            lines.append(f'Parameter Group {i}')
            for k in sorted(k for k in group if k != 'params'):
                lines.append(f'    {k}: {group[k]}')
        lines.append(')')
        return '\n'.join(lines)

# --- exercise it ---
t.manual_seed(0)
enc  = nn.Linear(4, 4)
head = nn.Linear(4, 2)
opt = MultiGroupSGD([
    {'params': enc.parameters(),  'lr': 0.001, 'weight_decay': 0.01},
    {'params': head.parameters(), 'lr': 0.01},
])

rep = repr(opt)
print(rep)

expected = (
    'MultiGroupSGD (\n'
    'Parameter Group 0\n'
    '    lr: 0.001\n'
    '    weight_decay: 0.01\n'
    'Parameter Group 1\n'
    '    lr: 0.01\n'
    ')'
)
assert rep == expected, f'Got:\n{rep}\n\nExpected:\n{expected}'
print('Multi-group repr correct!')